# artifacts: DINOv2 patch tokens + MIL

Проверяем, помогают ли локальные области шести рендеров находить `artifacts`. DINOv2 заморожен, обучается только маленькая голова. Нужен Colab с GPU.

Подготовлен, обучение MIL ещё не запускалось.

Порядок запуска: сверху вниз, в отдельном kernel. Данные и локальные модели нужны отдельно от Git. Установка и список файлов — в [инструкции](../../docs/RUNNING.md).

## 1. Данные

In [ ]:
%pip install -q iterative-stratification

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
assert torch.cuda.is_available(), 'В Colab включи GPU: Runtime → Change runtime type → T4 GPU'
device = torch.device('cuda')

from google.colab import drive
drive.mount('/content/drive')
paths = [Path.cwd(), *Path.cwd().parents,
         Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main'),
         Path('/content/drive/MyDrive/AI challenge main')]
ROOT = next((p for p in paths if (p / 'data/train.csv').is_file()), None)
import sys
sys.path.insert(0, str(ROOT))
from src.evaluation import binary_metrics, best_threshold
from src.data import split_indices, load_embeddings
from src.plots import setup_plot_style
setup_plot_style()
assert ROOT is not None, 'Не найдена папка проекта с data/train.csv'

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
train_idx, val_idx, test_idx = split_indices(labels)
train = labels.iloc[train_idx].reset_index(drop=True)
val = labels.iloc[val_idx].reset_index(drop=True)
objects = pd.concat([train, val], ignore_index=True)
assert (len(train), len(val), len(test_idx)) == (6259, 1361, 1344)
assert set(train.item_id).isdisjoint(val.item_id)
print('Train:', len(train), '| Validation:', len(val), '| artifacts:', int(val.artifacts.sum()))

TRAIN_DIR = ROOT / 'data/train'
if not (TRAIN_DIR / f'{objects.item_id.iloc[0]}.png').is_file():
    source_zip = ROOT / 'data/train_png.zip'
    assert source_zip.is_file(), source_zip
    local_zip = Path('/content/artifacts_train_png.zip')
    if not local_zip.is_file() or local_zip.stat().st_size != source_zip.stat().st_size:
        shutil.copy2(source_zip, local_zip)
    unpacked = Path('/content/artifacts_png')
    marker = unpacked / '.ready'
    if not marker.is_file():
        unpacked.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(local_zip) as archive:
            png_names = [name for name in archive.namelist() if name.lower().endswith('.png')]
            assert len(png_names) >= len(labels)
            for name in tqdm(png_names, desc='Распаковка PNG'):
                archive.extract(name, unpacked)
        marker.touch()
    first = next(unpacked.rglob(f'{objects.item_id.iloc[0]}.png'), None)
    assert first is not None
    TRAIN_DIR = first.parent
assert (TRAIN_DIR / f'{objects.item_id.iloc[-1]}.png').is_file()
print('PNG:', TRAIN_DIR)

## 2. Замороженный DINOv2

Делим общий PNG на шесть квадратов и уменьшаем каждый до 224×224, как в предыдущем опыте.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

def split_views(image):
    image = image.convert('RGB')
    width, height = image.size
    side = width // 3
    assert width == 3 * side and height == 2 * side
    return [image.crop((col * side, row * side, (col + 1) * side, (row + 1) * side))
            for row in range(2) for col in range(3)]

class Renders(Dataset):
    def __init__(self, positions):
        self.positions = list(positions)

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, index):
        position = self.positions[index]
        path = TRAIN_DIR / f'{objects.item_id.iloc[position]}.png'
        with Image.open(path) as image:
            views = torch.stack([transform(view) for view in split_views(image)])
        return position, views

revision = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
os.environ['XFORMERS_DISABLED'] = '1'
torch.hub.set_dir('/content/dinov2_hub')
dino = torch.hub.load(f'facebookresearch/dinov2:{revision}', 'dinov2_vits14',
                      pretrained=True, trust_repo=True, skip_validation=True)
dino = dino.to(device).eval()
dino.requires_grad_(False)

_, example = Renders([0])[0]
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    patch = dino.forward_features(example[:2].to(device))['x_norm_patchtokens']
assert patch.shape == (2, 256, 384), patch.shape
print('Patch tokens:', patch.shape)

## 3. Извлечение patch-признаков

Сжимаем сетку 16×16 до 8×8 областей на ракурс. Кэш около 2.2 ГБ лежит во временном хранилище Colab; результаты модели сохраняются в Drive.

In [ ]:
cache_dir = Path('/content/artifacts_mil_cache')
cache_dir.mkdir(parents=True, exist_ok=True)
token_path = cache_dir / 'patch_tokens.npy'
done_path = cache_dir / 'done.npy'
meta_path = cache_dir / 'meta.json'
shape = (len(objects), 6, 64, 384)
meta = {'revision': revision, 'shape': list(shape), 'seed': SEED,
        'item_ids_sha256': hashlib.sha256('\n'.join(objects.item_id).encode()).hexdigest()}

if (meta_path.is_file() and token_path.is_file() and done_path.is_file()
        and json.loads(meta_path.read_text()) == meta):
    tokens = np.load(token_path, mmap_mode='r+')
    done = np.load(done_path)
    assert tokens.shape == shape and done.shape == (len(objects),)
else:
    tokens = np.lib.format.open_memmap(token_path, mode='w+', dtype=np.float16, shape=shape)
    done = np.zeros(len(objects), dtype=bool)
    meta_path.write_text(json.dumps(meta))

view_batch_size = 8
missing = np.flatnonzero(~done)
loader = DataLoader(Renders(missing), batch_size=2, shuffle=False,
                    num_workers=2, pin_memory=True)
for positions, views in tqdm(loader, total=len(loader), desc='DINOv2 patches'):
    flat = views.reshape(-1, 3, 224, 224)
    blocks = []
    start = 0
    while start < len(flat):
        try:
            with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
                patch = dino.forward_features(flat[start:start + view_batch_size].to(device))['x_norm_patchtokens']
                patch = patch.transpose(1, 2).reshape(-1, 384, 16, 16)
                patch = F.adaptive_avg_pool2d(patch, (8, 8))
                patch = patch.flatten(2).transpose(1, 2)
            blocks.append(patch.cpu().numpy().astype(np.float16))
            start += len(patch)
        except torch.cuda.OutOfMemoryError:
            if view_batch_size == 1:
                raise
            view_batch_size = max(1, view_batch_size // 2)
            torch.cuda.empty_cache()
            print('Меньший batch ракурсов:', view_batch_size)
    block = np.concatenate(blocks).reshape(-1, 6, 64, 384)
    tokens[positions.numpy()] = block
    done[positions.numpy()] = True
    if done.sum() % 100 < len(positions):
        tokens.flush()
        np.save(done_path, done)

tokens.flush()
np.save(done_path, done)
assert done.all() and np.isfinite(tokens).all()
del dino
torch.cuda.empty_cache()
print('Готово:', tokens.shape, '| размер, ГБ:', round(token_path.stat().st_size / 1e9, 2))

## 4. MIL-голова

Каждый объект — набор из 384 областей: 6 ракурсов × 64 области. Сравниваем четыре способа объединить сигналы от областей.

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, positions):
        self.positions = np.asarray(positions)

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, index):
        position = self.positions[index]
        x = torch.from_numpy(np.asarray(tokens[position].reshape(384, 384), dtype=np.float32))
        y = torch.tensor(float(objects.artifacts.iloc[position]), dtype=torch.float32)
        return x, y

class MILHead(nn.Module):
    def __init__(self, pooling):
        super().__init__()
        self.pooling = pooling
        self.scorer = nn.Sequential(nn.LayerNorm(384), nn.Linear(384, 64),
                                    nn.GELU(), nn.Linear(64, 1))
        if pooling == 'attention':
            self.attention = nn.Sequential(nn.Linear(384, 64), nn.Tanh(), nn.Linear(64, 1))

    def forward(self, x):
        patch_scores = self.scorer(x).squeeze(-1)
        if self.pooling == 'mean':
            bag_score = patch_scores.mean(dim=1)
        elif self.pooling == 'max':
            bag_score = patch_scores.max(dim=1).values
        elif self.pooling == 'top8':
            bag_score = patch_scores.topk(8, dim=1).values.mean(dim=1)
        else:
            weights = torch.softmax(self.attention(x).squeeze(-1), dim=1)
            bag_score = (weights * patch_scores).sum(dim=1)
        heat = weights if self.pooling == 'attention' else patch_scores.sigmoid()
        return bag_score, heat

train_positions = np.arange(len(train))
val_positions = np.arange(len(train), len(objects))
val_loader = DataLoader(TokenDataset(val_positions), batch_size=64, shuffle=False)
positive = int(train.artifacts.sum())
pos_weight = torch.tensor((len(train) - positive) / positive, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print('Положительных в train:', positive, '| pos_weight:', round(pos_weight.item(), 2))

## 5. Обучение и сравнение на validation

In [ ]:
EPOCHS = 5
out = ROOT / 'labels/artifacts/models/mil'
out.mkdir(parents=True, exist_ok=True)
models = {}
results = []
probabilities = val[['item_id', 'artifacts']].copy()
y_val = val.artifacts.to_numpy()

for pooling in ['mean', 'max', 'top8', 'attention']:
    torch.manual_seed(SEED)
    head = MILHead(pooling).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    train_loader = DataLoader(TokenDataset(train_positions), batch_size=64, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED))
    for epoch in range(EPOCHS):
        head.train()
        losses = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, _ = head(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(pooling, 'epoch', epoch + 1, 'train loss', round(float(np.mean(losses)), 4))

    head.eval()
    p = []
    with torch.inference_mode():
        for x, _ in val_loader:
            logits, _ = head(x.to(device))
            p.append(logits.sigmoid().cpu().numpy())
    p = np.concatenate(p)
    threshold = float(max(np.arange(.01, 1, .01), key=lambda t: f1_score(y_val, p >= t)))
    prediction = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_val, prediction, labels=[0, 1]).ravel()
    row = {'pooling': pooling, 'F1': f1_score(y_val, prediction),
           'Precision': precision_score(y_val, prediction, zero_division=0),
           'Recall': recall_score(y_val, prediction, zero_division=0),
           'AP': average_precision_score(y_val, p),
           'FP': int(fp), 'FN': int(fn), 'threshold': threshold}
    results.append(row)
    probabilities[pooling] = p
    models[pooling] = head.cpu().eval()
    torch.save({'state_dict': models[pooling].state_dict(), 'pooling': pooling,
                'threshold': threshold, 'seed': SEED}, out / f'{pooling}.pth')
    print(pooling, '| F1:', round(row['F1'], 4), '| AP:', round(row['AP'], 4))

metrics = pd.DataFrame(results)
display(metrics.round(4))
metrics.to_csv(out / 'metrics.csv', index=False)
probabilities.to_csv(out / 'validation_probabilities.csv', index=False)
print('Предыдущий DINOv2 + geometry: F1 = 0.3717, AP = 0.2724')

## 6. Где модель увидела дефект

Для mean/max/top8 показываем оценку каждого patch; для attention — его вес. Вес не всегда означает положительный вклад в диагноз. Карта не является точной разметкой повреждённых полигонов.

In [ ]:
best_name = metrics.sort_values('F1', ascending=False).iloc[0]['pooling']
best = models[best_name].to(device).eval()
threshold = float(metrics.set_index('pooling').loc[best_name, 'threshold'])
review = val[['item_id', 'artifacts', 'noisy']].copy()
review['probability'] = probabilities[best_name].to_numpy()
review['prediction'] = (review.probability >= threshold).astype(int)

def show_attention(position, label):
    item_id = val.item_id.iloc[position]
    x, _ = TokenDataset([len(train) + position])[0]
    with torch.inference_mode():
        _, heat = best(x.unsqueeze(0).to(device))
    heat = heat[0].cpu().numpy().reshape(6, 8, 8)
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    with Image.open(TRAIN_DIR / f'{item_id}.png') as image:
        views = split_views(image)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(views[i])
        ax.imshow(heat[i], extent=(0, views[i].width, views[i].height, 0),
                  cmap='inferno', alpha=.45, interpolation='nearest')
        ax.set_title(f'Ракурс {i + 1}', color='#377FC8')
        ax.axis('off')
    fig.suptitle(f'{label}: {item_id} | p={review.probability.iloc[position]:.2f}')
    plt.tight_layout()
    plt.show()

for label, mask, ascending in [
    ('TP', (review.artifacts == 1) & (review.prediction == 1), False),
    ('FP', (review.artifacts == 0) & (review.prediction == 1), False),
    ('FN', (review.artifacts == 1) & (review.prediction == 0), True),
]:
    rows = review[mask].sort_values('probability', ascending=ascending)
    if len(rows):
        show_attention(rows.index[0], label)

## Вывод

Сравни F1 и AP с DINOv2 + geometry (0.3717 и 0.2724). Все четыре головы обучаются на одном train. Validation используется для сравнения вариантов и порогов, поэтому результат может быть оптимистичным. Internal test не трогаем.

## Как читать результат

F1 сравниваем на том же split. Порог и вариант модели выбираем по validation; internal test для подбора не используем. Повторный выбор по одной validation может завысить оценку. История и ограничения сохранены в [журнале](../../docs/experiments/artifacts.md). Формулы признаков — в [методике](../../docs/METHODS.md).